# 07. Extended VaR uplift -- 1h-vs-1d disagreement days (Table A.18, referee Q9)

Quantifies the economic value of the cross-frequency dissonance signal
in concrete VaR terms. Procedure:

1. Fit GMM regimes per frequency on the full sample.
2. Pool 5m returns within each regime (under the 1h classifier and the
   1d classifier separately) and compute the empirical 1% VaR per regime.
3. Aggregate per-day labels by mode and compute the population statistics
   of `max(VaR_1h, VaR_1d) / min(...)` on disagreement days.
4. Report the always-conservative average uplift relative to the 1d
   single-resolution baseline.

We run it in-process on `CL` and display the per-regime VaR
table together with the uplift summary.

**Re-run command**: `python run.py extended_var_uplift`

In [1]:
from __future__ import annotations
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, Markdown, display

PROJECT = Path.cwd()
_nb_file = globals().get("__vsc_ipynb_file__")
if _nb_file is not None:
    PROJECT = Path(_nb_file).resolve().parent.parent
elif PROJECT.name == "notebooks":
    PROJECT = PROJECT.parent
sys.path.insert(0, str(PROJECT))

OUT = PROJECT / "outputs"
OUT_2022 = PROJECT / "outputs_2022"
DATA = PROJECT / "data"
DATA_2022 = PROJECT / "data_2022"

pd.set_option("display.precision", 3)
pd.set_option("display.max_columns", 50)

## Step 1 -- load raw 5m OHLC for `CL`

Calls `src.data.data_ib.load_5m_ohlc` directly on `data/CL_5m.csv`. This is the same loader the pipeline uses; it parses mixed-offset DST timestamps as UTC then converts to NY local time.

In [2]:
from src.data.data_ib import load_5m_ohlc

FOCAL = "CL"
df_5m = load_5m_ohlc(DATA / f"{FOCAL}_5m.csv")
print(f"{FOCAL}_5m: {len(df_5m):,} bars, "
      f"{df_5m.index.min()} -> {df_5m.index.max()}, "
      f"{df_5m.index.normalize().nunique()} trading days")
df_5m.head()

CL_5m: 34,896 bars, 2025-11-02 18:00:00-05:00 -> 2026-05-01 16:55:00-04:00, 155 trading days


,Open,High,Low,Close,Volume
Date,,,,,
2025-11-02 18:00:00-05:00,58.36,58.43,58.10,58.29,2872.0
2025-11-02 18:05:00-05:00,58.30,58.37,58.29,58.35,669.0
2025-11-02 18:10:00-05:00,58.35,58.36,58.29,58.30,367.0
2025-11-02 18:15:00-05:00,58.30,58.33,58.26,58.33,394.0
2025-11-02 18:20:00-05:00,58.33,58.33,58.30,58.31,313.0


## Step 2 -- run the VaR-uplift analysis on the focal asset

In [3]:
from src.experiments.exp_06_var_uplift import var_uplift_1h_vs_1d

res = var_uplift_1h_vs_1d(df_5m, FOCAL, var_alpha=0.01)
pd.DataFrame([res])

C:\ProgramData\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=12.
  warnings.warn(
C:\ProgramData\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=12.
  warnings.warn(
C:\ProgramData\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=12.
  warnings.warn(
C:\ProgramData\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Win

,symbol,n_days_valid,n_disagree,pct_disagree,ratio_median,ratio_q75,ratio_q90,avg_uplift_pct_full_sample,avg_uplift_pct_disagree_only,trustworthy,degenerate_regimes,n_bars_1h_calm,n_bars_1h_crisis,n_bars_1d_calm,n_bars_1d_crisis
0,CL,155,73,47.097,2.209,2.209,2.209,23.497,2.176,True,,21743,13152,6216,28679


## Step 3 -- inspect the per-regime VaR table

Re-derives the per-regime VaR pool the experiment uses, so the reader can
see which regimes are pooled and how thin the tail samples are. A regime
with fewer than `min_regime_bars=50` 5m returns is flagged as degenerate
in the experiment output.

In [4]:
from src.core.models import fit_aligned_regimes
from src.core.config import FREQS

aligned = fit_aligned_regimes(df_5m, FOCAL, FREQS)
rets = np.log(df_5m["Close"] / df_5m["Close"].shift(1)).replace([np.inf, -np.inf], np.nan)
df_bars = pd.DataFrame({
    "r": rets,
    "lab1h": aligned["1h"].reindex(rets.index).astype(float),
    "lab1d": aligned["1d"].reindex(rets.index).astype(float),
}).dropna()
rows = []
for label_col, key in (("lab1h", "1h"), ("lab1d", "1d")):
    for g in sorted(df_bars[label_col].unique()):
        sub = df_bars.loc[df_bars[label_col] == g, "r"].values
        rows.append({
            "freq": key, "regime": int(g), "n_bars": len(sub),
            "var_1pct": float(np.quantile(sub, 0.01)) if len(sub) >= 50 else float("nan"),
        })
pd.DataFrame(rows)

C:\ProgramData\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=12.
  warnings.warn(
C:\ProgramData\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=12.
  warnings.warn(
C:\ProgramData\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=12.
  warnings.warn(
C:\ProgramData\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Win

,freq,regime,n_bars,var_1pct
0,1h,0,21743,-0.003
1,1h,1,13152,-0.011
2,1d,0,6216,-0.008
3,1d,1,28679,-0.007


## Cached all-asset summary -- `outputs/var_uplift_by_resolution.csv`

In [5]:
p = OUT / "var_uplift_by_resolution.csv"
display(pd.read_csv(p).round(3) if p.exists() else Markdown(f"`{p}` missing"))

,symbol,n_days_valid,n_disagree,pct_disagree,ratio_median,ratio_q75,ratio_q90,avg_uplift_pct_full_sample,avg_uplift_pct_disagree_only,trustworthy,degenerate_regimes,n_bars_1h_calm,n_bars_1h_crisis,n_bars_1d_calm,n_bars_1d_crisis
0,SPY,124,40,32.258,1.350,1.350,1.384,10.173,31.536,True,NaN,6900,16644,12024,11520
1,USDJPY,156,77,49.359,1.161,1.161,1.161,7.592,0.775,True,NaN,30471,5913,12471,23913
2,CL,155,73,47.097,2.209,2.209,2.209,23.497,2.176,True,NaN,21743,13152,6216,28679
3,GLD,124,74,59.677,1.561,1.561,1.561,15.809,4.146,True,NaN,15228,8316,2112,21432
